# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [1]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.17 | NumPy 2.3.3 | pandas 3.0.5
Wrote 5 data files to /Users/saraadelinazavala/Library/CloudStorage/OneDrive2-UCSF/DATASCI217/ds217-26f-06/data
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

**TODO:** Replace this text with an evidence map: one bullet for each of Lectures 01 to 06. For 01, 02, 04, 05, and 06, name one concrete technique or file and say in one sentence where this notebook or its outputs use it. For 03, complete the NumPy calculation after Task 4.3 and explain your axis choice in one sentence here. The lectures cover the command line and Python basics (01); Git, functions, and files (02); environments, shell pipelines, and NumPy (03); Jupyter and labeled pandas data (04); data contracts, cleaning, and validation (05); and joins, reshaping, and concatenation (06). Do not paste earlier solutions.

## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [2]:
%%bash
# paste your command here%%bash
wc -l data/*.csv > output/file_inventory.txt

### 1.2 Profile each table with a function

In [3]:
def profile_table(name, table):
    return {
        "table": name,
        "rows": len(table),
        "columns": len(table.columns),
        "exact_duplicate_rows": int(table.duplicated().sum()),
    }


results = []

for name in manifest["files"]:
    table = raw[Path(name).stem]
    results.append(profile_table(name, table))

table_profile = pd.DataFrame(results)

table_profile.to_csv(OUTPUT_DIR / "table_profile.csv", index=False)

display(table_profile)

,table,rows,columns,exact_duplicate_rows
0,patients.parquet,30045,4,45
1,sites.csv,5,3,0
2,visits_north.csv,26297,6,35
3,visits_south.csv,31074,6,35
4,a1c_wide.parquet,29424,3,0


## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)
### Cleaned table grains and keys

- **patients:** One row per registered patient. The key is `patient_id`.
  Exact duplicate rows in the raw table violate this intended key.

- **sites:** One row per current site. The key is `site_code`.
  Current and retired records for the same site repeat this key in the raw table.
  Keep only current records.

- **visits_north and visits_south:** After stacking and cleaning, one row
  per patient and visit, keeping the later reading when there is a recheck.
  The composite key is (`patient_id`, `visit`).
  Exact duplicate rows and rechecks violate this intended key in the raw data.

- **a1c_wide:** One row per patient's lab results, with separate baseline
  and week12 columns. The key is `patient_id`.
  After reshaping to long form, the key is (`patient_id`, `visit`).

### Expected join cardinalities

- Visits to patients: many-to-one on `patient_id`, because a patient can
  have multiple visits but only one registry row.
- Patients to current sites: many-to-one on `site_code`, because multiple
  patients can belong to the same site.
- Visits to long-form A1c results: one-to-one on (`patient_id`, `visit`),
  because each cleaned table has at most one row per patient and visit.

**TODO:** Replace this text with the contract: the intended grain and key (single or composite) of each cleaned input table, and the cardinality you expect for each Task 4.4 join. Describe visits after stacking and keeping the later recheck, sites after keeping current records, and the long-form A1c join keys. Note raw exact repeats, rechecks, and site versions that violate the intended cleaned keys (README Task 2.1).

### 2.2 Stack the two visit files

In [7]:
north = raw["visits_north"].copy(deep=True)
south = raw["visits_south"].copy(deep=True)

south = south.rename(columns={"wt_kg": "weight_kg"})

north["source_file"] = "visits_north.csv"
south["source_file"] = "visits_south.csv"

visits_raw = pd.concat([north, south], ignore_index=True)

print("Shape:", visits_raw.shape)
display(visits_raw.head())

Shape: (57371, 7)


,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
0,P00002,baseline,2026-02-20,09:50,132,85.2,visits_north.csv
1,P00002,week12,2026-05-08,15:20,129,82.0,visits_north.csv
2,P00003,baseline,2026-01-16,11:45,148,67.5,visits_north.csv
3,P00003,wk12,2026-04-07,10:00,137,64.0,visits_north.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv


### 2.3 Audit the raw tables

In [8]:
# Temporary variables for auditing; raw tables stay unchanged.
patients_audit = raw["patients"].copy(deep=True)
sites_audit = raw["sites"].copy(deep=True)

# Normalize visit labels only to identify repeated patient/visit pairs.
visit_map = {
    "baseline": "baseline",
    "bl": "baseline",
    "week12": "week12",
    "week 12": "week12",
    "wk12": "week12",
}

visit_labels = visits_raw["visit"].str.strip().str.lower().map(visit_map)
assert visit_labels.notna().all(), "An unexpected visit label needs checking."

visit_pairs = visits_raw[["patient_id"]].copy(deep=True)
visit_pairs["visit"] = visit_labels

# Check the site-code normalization rule.
normalized_sites = (
    patients_audit["site_code"]
    .str.upper()
    .str.replace(" ", "", regex=False)
    .str.replace("-", "", regex=False)
)

# Convert blood pressure temporarily for the range check.
sbp = pd.to_numeric(visits_raw["sbp_mmhg"], errors="coerce")

# Require both exact YYYY-MM-DD formatting and a valid calendar date.
date_text = visits_raw["visit_date"]
exact_format = date_text.str.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}")

parsed_dates = pd.to_datetime(
    date_text.where(exact_format),
    format="%Y-%m-%d",
    errors="coerce",
)

empty_gender = patients_audit["gender"].str.strip().eq("")

issue_audit = pd.DataFrame({
    "issue": [
        "patients: gender values that are not a code",
        "patients: empty gender",
        "patients: birth_year sentinel 9999",
        "patients: site codes needing normalization",
        "sites: rows with a repeated site_code",
        "visits: visit labels that are not baseline or week12",
        "visits: sbp sentinel 999",
        "visits: sbp outside 70 through 250",
        "visits: date parse failures",
        "visits: rows in repeated patient_id and visit pairs",
    ],
    "count": [
        (
            ~empty_gender
            & ~patients_audit["gender"].isin(["F", "M", "X"])
        ).sum(),
        empty_gender.sum(),
        patients_audit["birth_year"].eq("9999").sum(),
        patients_audit["site_code"].ne(normalized_sites).sum(),
        sites_audit.duplicated(
            subset=["site_code"], keep=False
        ).sum(),
        (~visits_raw["visit"].isin(["baseline", "week12"])).sum(),
        visits_raw["sbp_mmhg"].eq("999").sum(),
        (sbp.ne(999) & (sbp.lt(70) | sbp.gt(250))).sum(),
        parsed_dates.isna().sum(),
        visit_pairs.duplicated(
            subset=["patient_id", "visit"], keep=False
        ).sum(),
    ],
})

issue_audit["count"] = issue_audit["count"].astype("int64")
issue_audit.to_csv(OUTPUT_DIR / "issue_audit.csv", index=False)

display(issue_audit)

assert all(
    raw[name].equals(raw_snapshot[name]) for name in raw
), "Auditing must not change raw."


,issue,count
0,patients: gender values that are not a code,6968
1,patients: empty gender,486
2,patients: birth_year sentinel 9999,371
3,patients: site codes needing normalization,1811
4,sites: rows with a repeated site_code,4
5,visits: visit labels that are not baseline or ...,2309
6,visits: sbp sentinel 999,341
7,visits: sbp outside 70 through 250,116
8,visits: date parse failures,169
9,visits: rows in repeated patient_id and visit ...,380


## Task 3: Clean

### 3.1 Clean the patients

In [10]:
# Copy the patients table and remove exact duplicates.
patients_clean = raw["patients"].copy(deep=True)
patients_clean = patients_clean.drop_duplicates(keep="first")

# Show the original gender spellings.
display(patients_clean["gender"].value_counts(dropna=False))

# Map known spellings to codes; blanks and unknown become missing.
gender_map = {
    "f": "F",
    "female": "F",
    "woman": "F",
    "m": "M",
    "male": "M",
    "man": "M",
    "x": "X",
    "nonbinary": "X",
    "non-binary": "X",
    "nb": "X",
}

patients_clean["gender"] = (
    patients_clean["gender"]
    .str.strip()
    .str.lower()
    .map(gender_map)
    .astype("category")
)

# Replace the birth-year sentinel with missing; use nullable integers.
birth_year = pd.to_numeric(
    patients_clean["birth_year"], errors="coerce"
)

patients_clean["birth_year"] = (
    birth_year.mask(birth_year.eq(9999)).astype("Int64")
)

# Normalize site codes.
patients_clean["site_code"] = (
    patients_clean["site_code"]
    .str.upper()
    .str.replace(" ", "", regex=False)
    .str.replace("-", "", regex=False)
)

# Create and save the required summary.
patients_summary = pd.DataFrame({
    "metric": [
        "patients",
        "birth_year missing",
        "birth_year min",
        "birth_year max",
        "site_code values",
        "patients at N01",
        "patients at S02",
        "patients at E04",
    ],
    "value": [
        len(patients_clean),
        patients_clean["birth_year"].isna().sum(),
        patients_clean["birth_year"].min(),
        patients_clean["birth_year"].max(),
        patients_clean["site_code"].nunique(),
        patients_clean["site_code"].eq("N01").sum(),
        patients_clean["site_code"].eq("S02").sum(),
        patients_clean["site_code"].eq("E04").sum(),
    ],
})

patients_summary.to_csv(
    OUTPUT_DIR / "patients_summary.csv", index=False
)



gender
F             11230
M             10952
male           1038
female         1018
f               622
Male            621
Female          577
m               537
woman           399
                381
X               372
man             329
unknown         282
MALE            255
FEMALE          231
 M              223
 F              205
Woman           131
nonbinary       118
                104
NB              103
Unknown          96
non-binary       80
x                62
Non-Binary       34
Name: count, dtype: int64[pyarrow]

In [11]:
# Save category counts, including missing values.
gender_counts = (
    patients_clean["gender"]
    .value_counts(dropna=False)
    .reset_index()
)
gender_counts["code"] = gender_counts["gender"].cat.codes

gender_counts.to_csv(
    OUTPUT_DIR / "gender_categories.csv", index=False
)

display(patients_summary)
display(gender_counts)

assert patients_clean["patient_id"].is_unique
assert gender_counts["count"].sum() == len(patients_clean)
assert all(raw[name].equals(raw_snapshot[name]) for name in raw)

,metric,value
0,patients,30000
1,birth_year missing,371
2,birth_year min,1941
3,birth_year max,1996
4,site_code values,3
5,patients at N01,13739
6,patients at S02,15706
7,patients at E04,555


,gender,count,code
0,F,14413,0
1,M,13955,1
2,NaN,863,-1
3,X,769,2


### 3.2 Clean the visits

In [12]:
visits_clean = visits_raw.copy(deep=True)

# 1. Remove exact duplicates.
visits_clean = visits_clean.drop_duplicates(keep="first")

# 2. Display and standardize visit labels.
display(visits_clean["visit"].value_counts(dropna=False))

visit_map = {
    "baseline": "baseline",
    "bl": "baseline",
    "week12": "week12",
    "week 12": "week12",
    "wk12": "week12",
}

visits_clean["visit"] = (
    visits_clean["visit"]
    .str.strip()
    .str.lower()
    .map(visit_map)
)

# 3. Keep blood pressure readings from 70 through 250.
sbp = pd.to_numeric(visits_clean["sbp_mmhg"], errors="coerce")

visits_clean["sbp_mmhg"] = (
    sbp.where(sbp.between(70, 250)).astype("Int64")
)

# 4. Convert weight to numbers; blanks become missing.
visits_clean["weight_kg"] = pd.to_numeric(
    visits_clean["weight_kg"], errors="coerce"
).astype("float64")

# 5. Accept only exact YYYY-MM-DD text and valid calendar dates.
date_text = visits_clean["visit_date"]
exact_format = date_text.str.fullmatch(
    r"[0-9]{4}-[0-9]{2}-[0-9]{2}"
)

visits_clean["visit_date"] = pd.to_datetime(
    date_text.where(exact_format),
    format="%Y-%m-%d",
    errors="coerce",
)

display(visits_clean.head())

visit
baseline     28496
week12       26498
BASELINE       255
Baseline       242
bl             235
baseline       228
BL             220
wk12           204
 week12        198
Week 12        194
WEEK12         186
Wk12           177
week 12        168
Name: count, dtype: int64[pyarrow]

,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
0,P00002,baseline,2026-02-20,09:50,132,85.2,visits_north.csv
1,P00002,week12,2026-05-08,15:20,129,82.0,visits_north.csv
2,P00003,baseline,2026-01-16,11:45,148,67.5,visits_north.csv
3,P00003,week12,2026-04-07,10:00,137,64.0,visits_north.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv


In [13]:
# Show every row belonging to a repeated patient/visit pair.
repeated_pairs = visits_clean.duplicated(
    subset=["patient_id", "visit"],
    keep=False,
)

display(
    visits_clean.loc[repeated_pairs].sort_values(
        ["patient_id", "visit", "visit_time"]
    )
)

# Sort by time so the last row is the later reading.
visits_clean = (
    visits_clean
    .sort_values("visit_time")
    .drop_duplicates(
        subset=["patient_id", "visit"],
        keep="last",
    )
)

assert not visits_clean.duplicated(
    subset=["patient_id", "visit"]
).any()

,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
26298,P00001,baseline,2026-03-06,14:15,141,52.4,visits_south.csv
26297,P00001,baseline,2026-03-06,14:40,146,52.4,visits_south.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv
5,P00004,baseline,2026-02-25,11:15,142,106.2,visits_north.csv
84,P00088,baseline,2026-03-10,09:30,153,68.2,visits_north.csv
...,...,...,...,...,...,...,...
55473,P28232,baseline,2026-01-12,10:10,159,61.0,visits_south.csv
56249,P28953,baseline,2026-01-21,10:30,153,74.8,visits_south.csv
56248,P28953,baseline,2026-01-21,10:45,151,74.8,visits_south.csv
25464,P29170,baseline,2026-03-02,11:20,130,78.0,visits_north.csv


In [14]:
visits_clean = (
    visits_clean
    .sort_values(["patient_id", "visit"])
    .reset_index(drop=True)
)

visits_clean = visits_clean[
    [
        "patient_id",
        "visit",
        "visit_date",
        "sbp_mmhg",
        "weight_kg",
        "source_file",
    ]
]

visits_summary = pd.DataFrame({
    "metric": [
        "visits",
        "baseline visits",
        "week12 visits",
        "visit_date missing",
        "visit_date min",
        "visit_date max",
        "sbp_mmhg missing",
        "sbp_mmhg min",
        "sbp_mmhg max",
        "weight_kg missing",
        "weight_kg mean",
        "visits from visits_north.csv",
        "visits from visits_south.csv",
    ],
    "value": [
        len(visits_clean),
        visits_clean["visit"].eq("baseline").sum(),
        visits_clean["visit"].eq("week12").sum(),
        visits_clean["visit_date"].isna().sum(),
        visits_clean["visit_date"].min(),
        visits_clean["visit_date"].max(),
        visits_clean["sbp_mmhg"].isna().sum(),
        visits_clean["sbp_mmhg"].min(),
        visits_clean["sbp_mmhg"].max(),
        visits_clean["weight_kg"].isna().sum(),
        round(visits_clean["weight_kg"].mean(), 2),
        visits_clean["source_file"].eq("visits_north.csv").sum(),
        visits_clean["source_file"].eq("visits_south.csv").sum(),
    ],
})

visits_summary.to_csv(VISITS_SUMMARY_PATH, index=False)

display(visits_summary)
display(visits_clean["visit"].value_counts(dropna=False))

assert visits_clean["visit"].isin(["baseline", "week12"]).all()
assert all(raw[name].equals(raw_snapshot[name]) for name in raw)

,metric,value
0,visits,57181
1,baseline visits,29599
2,week12 visits,27582
3,visit_date missing,169
4,visit_date min,2026-01-05 00:00:00
5,visit_date max,2026-07-06 00:00:00
6,sbp_mmhg missing,456
7,sbp_mmhg min,70
8,sbp_mmhg max,250
9,weight_kg missing,828


visit
baseline    29599
week12      27582
Name: count, dtype: int64

### 3.3 Record the decisions

In [15]:
decision_table = pd.DataFrame(
    [
        [
            "patients",
            "exact duplicate rows",
            "Removed exact duplicates and kept the first copy.",
            "Repeated copies represent the same patient and should not be counted twice.",
        ],
        [
            "patients",
            "gender spellings that are not a code",
            "Stripped spaces, lowercased spellings, mapped them to F, M, or X, and converted to category.",
            "Equivalent spellings should share one code, while blank and unknown values remain missing.",
        ],
        [
            "patients",
            "site codes needing normalization",
            "Uppercased site codes and removed all spaces and hyphens.",
            "Formatting differences should not prevent records for the same site from matching.",
        ],
        [
            "patients",
            "birth_year sentinel 9999",
            "Replaced 9999 with missing and converted birth_year to Int64.",
            "9999 means the birth year was not recorded and is not an actual birth year.",
        ],
        [
            "visits",
            "weight column named wt_kg in visits_south.csv",
            "Renamed wt_kg to weight_kg before stacking the visit files.",
            "Both columns contain weight in kilograms and need the same name to stack correctly.",
        ],
        [
            "visits",
            "sbp sentinel 999 and readings outside 70 through 250",
            "Kept readings from 70 through 250 inclusive and replaced all other values with missing.",
            "999 means not measured, and values outside the specified range violate the data contract.",
        ],
        [
            "visits",
            "impossible and loosely written visit dates",
            "Parsed only exact YYYY-MM-DD text representing valid calendar dates and made other values missing.",
            "The contract requires both the specified date format and a date that exists.",
        ],
        [
            "visits",
            "repeated patient_id and visit pairs",
            "Removed exact duplicates, then sorted by visit_time and kept the later reading for each pair.",
            "Rechecks belong to the same patient visit, and the assignment specifies keeping the later measurement.",
        ],
        [
            "sites",
            "repeated site_code rows",
            "Keep only rows with record_status equal to current when creating sites_current in Task 4.2.",
            "Retired site records would create duplicate matches and introduce outdated site names.",
        ],
    ],
    columns=["table", "issue", "decision", "reason"],
)

decision_table.to_csv(
    OUTPUT_DIR / "decision_log.csv",
    index=False,
)

display(decision_table)

,table,issue,decision,reason
0,patients,exact duplicate rows,Removed exact duplicates and kept the first copy.,Repeated copies represent the same patient and...
1,patients,gender spellings that are not a code,"Stripped spaces, lowercased spellings, mapped ...","Equivalent spellings should share one code, wh..."
2,patients,site codes needing normalization,Uppercased site codes and removed all spaces a...,Formatting differences should not prevent reco...
3,patients,birth_year sentinel 9999,Replaced 9999 with missing and converted birth...,9999 means the birth year was not recorded and...
4,visits,weight column named wt_kg in visits_south.csv,Renamed wt_kg to weight_kg before stacking the...,Both columns contain weight in kilograms and n...
5,visits,sbp sentinel 999 and readings outside 70 throu...,Kept readings from 70 through 250 inclusive an...,"999 means not measured, and values outside the..."
6,visits,impossible and loosely written visit dates,Parsed only exact YYYY-MM-DD text representing...,The contract requires both the specified date ...
7,visits,repeated patient_id and visit pairs,"Removed exact duplicates, then sorted by visit...","Rechecks belong to the same patient visit, and..."
8,sites,repeated site_code rows,Keep only rows with record_status equal to cur...,Retired site records would create duplicate ma...


## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [ ]:
# TODO: melt raw["a1c_wide"], map the visit labels, drop the rows with no result,
# then save the per-visit summary to A1C_SUMMARY_PATH (README Task 4.1).
a1c_long = None

### 4.2 Check the keys before joining

In [ ]:
# TODO: steps 1 and 2, catch the MergeError, show the repeated site rows, and build sites_current.
sites = raw["sites"]
sites_current = None

In [ ]:
# TODO: step 3, outer merge visits_clean and a1c_long with indicator=True and show the _merge counts,
# then save the seven counts in README Task 4.2 to MERGE_AUDIT_PATH.

### 4.3 Pivot blood pressure to one row per patient

In [ ]:
# TODO: pivot visits_clean, reset the index, clear columns.name, add sbp_change, then save the
# SAMPLE_IDS rows to SBP_WIDE_SAMPLE_PATH (README Task 4.3).
sbp_wide = None

#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [ ]:
# TODO: select complete pairs, convert to a NumPy array, and display its shape and two per-visit means.

### 4.4 Join the analysis table

In [ ]:
# TODO: three left merges onto visits_clean, each with validate=, then the columns and dtypes
# in README Task 4.4. A MergeError means an earlier step left a problem: see HINTS.md.
analysis = None

## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [ ]:
# TODO: collect named True/False checks in a Series (README Task 5.1), display it, and stop with
# assert checks.all(), checks[~checks] before the analysis extract is saved.
# A failed check means an earlier step left a problem: print checks[~checks] and see HINTS.md.

### 5.2 Save the analysis extract

In [ ]:
# TODO: keep the analysis rows whose patient_id is in SAMPLE_IDS, renumber them, and save them to
# ANALYSIS_SAMPLE_PATH with index=False (README Task 5.2).

### 5.3 Read the file back

In [ ]:
# TODO: read ANALYSIS_SAMPLE_PATH back with its dtypes and parse_dates=["visit_date"], and print
# whether it equals analysis_sample.